<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/02_run_asr_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import librosa
import pandas as pd
import torch
import soundfile as sf
from google.colab import drive
from transformers import Wav2Vec2ForCTC, AutoProcessor

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

samples_folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/samples_20")
samples = pd.read_csv(samples_folder / "samples_20.csv", dtype=str)
print(len(samples), "samples")
samples.head()

20 samples


,file_id,speaker_id,text,audio_file
0,0f5934e285,f6b2d,समेत पाइन्छ,0f5934e285.flac
1,0a4ae83ed0,29944,विश्वको अर्थतन्त्रलाई असर,0a4ae83ed0.flac
2,0a0f56342e,0cc60,एमएस्सीका लागि,0a0f56342e.flac
3,0eb1d8345a,d4c0f,जिरालाई पानीमा पिनेर,0eb1d8345a.flac
4,08f8ed5eeb,0b298,तर विडम्बना के छ,08f8ed5eeb.flac


In [3]:
model_name = "facebook/mms-1b-all"
processor = AutoProcessor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

In [4]:
# language codes that start with n, Nepali should be in this list
all_languages = list(processor.tokenizer.vocab.keys())
print([lang for lang in all_languages if lang.startswith("n")])

['nab', 'nag', 'nan', 'nas', 'naw', 'nca', 'nch', 'ncj', 'ncl', 'ncu', 'ndj', 'ndp', 'ndv', 'ndy', 'ndz', 'neb', 'new', 'nfa', 'nfr', 'nga', 'ngl', 'ngp', 'ngu', 'nhe', 'nhi', 'nhu', 'nhw', 'nhx', 'nhy', 'nia', 'nij', 'nim', 'nin', 'nko', 'nlc', 'nld', 'nlg', 'nlk', 'nmz', 'nnb', 'nno', 'nnq', 'nnw', 'noa', 'nob', 'nod', 'nog', 'not', 'npi', 'npl', 'npy', 'nso', 'nst', 'nsu', 'ntm', 'ntr', 'nuj', 'nus', 'nuz', 'nwb', 'nxq', 'nya', 'nyf', 'nyn', 'nyo', 'nyy', 'nzi']


In [5]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using:", device)

using: cuda


In [6]:
processor.tokenizer.set_target_lang("npi")
model.load_adapter("npi")
model = model.to(device)
model.eval()
print("model ready")

adapter.npi.safetensors: reconstructing file:   0%|          |  0.00B / 9.28MB            

adapter.npi.safetensors: downloading bytes:           |  0.00B            

model ready


## Get the model text for one audio file
This function reads the audio and gives back the text from the model.

In [7]:
def get_prediction(audio_file):
    audio, sample_rate = sf.read(audio_file)
    model_input = processor(audio, sampling_rate=sample_rate, return_tensors="pt").to(device)
    with torch.no_grad():
        scores = model(**model_input).logits
    best_ids = torch.argmax(scores, dim=-1)[0]
    return processor.decode(best_ids)

## Test with the first file

In [8]:
first_file = samples_folder / samples.loc[0, "audio_file"]
print("correct text:", samples.loc[0, "text"])
print("model text:  ", get_prediction(first_file))

correct text: समेत पाइन्छ
model text:   समेत पाइन्छ


## Run all 20 files and save
Saved in Google Drive so we can use it tomorrow.

In [9]:
all_predictions = []
for audio_name in samples["audio_file"]:
    all_predictions.append(get_prediction(samples_folder / audio_name))

samples["mms_prediction"] = all_predictions
samples.to_csv(samples_folder / "predictions_mms.csv", index=False)
samples[["text", "mms_prediction"]].head(10)

,text,mms_prediction
0,समेत पाइन्छ,समेत पाइन्छ
1,विश्वको अर्थतन्त्रलाई असर,विश्वको अर्थतन्त्रलाई असर
2,एमएस्सीका लागि,kanananananana aeaeaeaeaeanananr
3,जिरालाई पानीमा पिनेर,जिरालाई पानीमा पिनेर
4,तर विडम्बना के छ,तर विडमबना के छ
5,नेपालमा बोलिने,नेपालमा बोल्लिने
6,गर्न छोडेको,गर्न छोटिेको
7,झुकाएर तीर छोड्दछौँ,झुकाएर तिर छोड्द छौँ
8,अधिक भारी बाउन्ने,अधिक भारी बाउने
9,यसै कडीको भाग थिए,यसै गढिको भाग थिए
